In [1]:
# step01_eda.py
# Run in the project root where Cleaned_GPS_Spoofing_Dataset.csv is located.

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# -------------- Config --------------
DATA_PATH = "Cleaned_GPS_Spoofing_Dataset.csv"
OUTPUT_DIR = "eda_outputs"
RANDOM_STATE = 42
# ------------------------------------

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1) Load dataset
df = pd.read_csv(DATA_PATH)
print("Loaded:", DATA_PATH)
print("Shape:", df.shape)

Loaded: Cleaned_GPS_Spoofing_Dataset.csv
Shape: (510530, 14)


In [2]:
# 2) Quick peek
print("\n--- Head ---")
print(df.head())
print("\n--- Info ---")
df.info()


--- Head ---
   PRN           DO             PD         RX            TOW            CP  \
0    6  1160.671408 -238359.543282  491568.00  491568.000795 -24660.263293   
1    6  1157.827808 -238363.921192  491568.02  491568.020795 -24683.468520   
2    6  1161.135932 -238368.638615  491568.04  491568.040795 -24706.687357   
3    6  1161.156617 -238373.391775  491568.06  491568.060795 -24729.920039   
4    6  1160.690893 -238378.127066  491568.08  491568.080795 -24753.153999   

          EC         LC         PC        PIP          PQP        TCD  \
0  163521.78  170008.81  179294.97  178420.33 -17688.17800  1158.4806   
1  158277.05  160253.09  182106.11  182106.05    158.33333  1160.7948   
2  184442.36  193125.89  208228.02 -205198.30  35391.59000  1157.6909   
3  159812.45  169960.55  187550.72  187543.00  -1701.94400  1161.1343   
4  183557.66  191912.58  208403.50  208330.05  -5532.89990  1161.1759   

         CN0  Output  
0  49.412529       0  
1  49.452686       0  
2  49.480

In [3]:
# 3) Missing values and duplicates
print("\n--- Missing values ---")
print(df.isna().sum())
print("\n--- Duplicates ---")
print("Duplicate rows:", df.duplicated().sum())


--- Missing values ---
PRN       0
DO        0
PD        0
RX        0
TOW       0
CP        0
EC        0
LC        0
PC        0
PIP       0
PQP       0
TCD       0
CN0       0
Output    0
dtype: int64

--- Duplicates ---
Duplicate rows: 31218


In [4]:
# 4) Target distribution
if 'Output' not in df.columns:
    raise ValueError("Expected a column named 'Output' as target.")
dist = df['Output'].value_counts().sort_index()
print("\n--- Target distribution ---")
print(dist)
dist.plot(kind='bar')
plt.title('Target class distribution')
plt.xlabel('Output')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "target_distribution.png"))
plt.clf()


--- Target distribution ---
Output
0    397825
1     36458
2     44232
3     32015
Name: count, dtype: int64


<Figure size 640x480 with 0 Axes>

In [5]:
# 5) Summary statistics (save)
summary = df.describe().T
summary.to_csv(os.path.join(OUTPUT_DIR, "summary_statistics.csv"))
print("\nSaved summary_statistics.csv")


Saved summary_statistics.csv


In [6]:
# 6) Correlation matrix (heatmap)
# Use numeric columns only
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
# Remove target from correlation display
if 'Output' in num_cols:
    num_cols_no_target = [c for c in num_cols if c != 'Output']
else:
    num_cols_no_target = num_cols

corr = df[num_cols_no_target].corr()
plt.figure(figsize=(12,10))
sns.heatmap(corr, annot=False, cmap="coolwarm", linewidths=0.3)
plt.title("Feature correlation heatmap")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "correlation_heatmap.png"))
plt.clf()
print("Saved correlation_heatmap.png")

Saved correlation_heatmap.png


<Figure size 1200x1000 with 0 Axes>

In [7]:
# 7) Pairwise scatter sample (for a few important numeric pairs)
sample_cols = num_cols_no_target[:6]  # first 6 numeric features, adjust if needed
sns.pairplot(df[sample_cols + ['Output']].sample(frac=0.02, random_state=RANDOM_STATE), 
             hue='Output', diag_kind='kde', plot_kws={'alpha':0.6, 's':10})
plt.savefig(os.path.join(OUTPUT_DIR, "pairplot_sample.png"))
plt.clf()
print("Saved pairplot_sample.png")

Saved pairplot_sample.png


<Figure size 1559.36x1500 with 0 Axes>

In [8]:
# 8) Feature distributions & boxplots
for col in num_cols_no_target:
    plt.figure(figsize=(6,3))
    sns.histplot(df[col], bins=80, kde=True)
    plt.title(f"Distribution: {col}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"dist_{col}.png"))
    plt.clf()
    
    plt.figure(figsize=(6,2))
    sns.boxplot(x=df[col])
    plt.title(f"Boxplot: {col}")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"box_{col}.png"))
    plt.clf()

print("Saved individual feature dist/box plots")

C:\Users\anmol\AppData\Local\Temp\ipykernel_2284\2737545223.py:3: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  plt.figure(figsize=(6,3))


Saved individual feature dist/box plots


<Figure size 600x300 with 0 Axes>

<Figure size 600x200 with 0 Axes>

<Figure size 600x300 with 0 Axes>

...

<Figure size 600x200 with 0 Axes>

<Figure size 600x300 with 0 Axes>

<Figure size 600x200 with 0 Axes>

<Figure size 600x300 with 0 Axes>

<Figure size 600x200 with 0 Axes>

In [9]:
# 9) Quick feature importance proxy using RandomForest (fast)
# This is only a proxy to locate candidate features for modelling.
X = df.drop(columns=['Output'])
y = df['Output']

# small train subset to get importance quickly
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.8, random_state=RANDOM_STATE, stratify=y)
rf_quick = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1)
rf_quick.fit(X_train, y_train)

importances = pd.Series(rf_quick.feature_importances_, index=X.columns).sort_values(ascending=False)
importances.to_csv(os.path.join(OUTPUT_DIR, "rf_quick_feature_importances.csv"))
print("\nTop features (RandomForest proxy):")
print(importances.head(20))

# Plot importances
plt.figure(figsize=(8,6))
sns.barplot(x=importances.values[:30], y=importances.index[:30])
plt.title("Feature importances (RandomForest proxy)")
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "feature_importances.png"))
plt.clf()
print("Saved feature_importances.png")


Top features (RandomForest proxy):
TOW    0.178505
PD     0.169958
RX     0.169271
TCD    0.095072
CP     0.094716
DO     0.092715
PRN    0.060686
CN0    0.056239
PC     0.018740
LC     0.018320
EC     0.017127
PQP    0.014862
PIP    0.013790
dtype: float64
Saved feature_importances.png


<Figure size 800x600 with 0 Axes>

In [10]:
# 10) Correlation of top features with target (boxplots)
top_feats = importances.index[:8].tolist()
for f in top_feats:
    plt.figure(figsize=(6,3))
    sns.boxplot(x=df['Output'], y=df[f])
    plt.title(f"{f} by Output")
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f"box_by_target_{f}.png"))
    plt.clf()

print("Saved box_by_target for top features")

Saved box_by_target for top features


<Figure size 600x300 with 0 Axes>

<Figure size 600x300 with 0 Axes>

<Figure size 600x300 with 0 Axes>

<Figure size 600x300 with 0 Axes>

<Figure size 600x300 with 0 Axes>

<Figure size 600x300 with 0 Axes>

<Figure size 600x300 with 0 Axes>

<Figure size 600x300 with 0 Axes>

In [11]:
# 11) Save cleaned sample CSV of top features for quick experiments
selected_features = top_feats + ['Output']
df[selected_features].to_csv(os.path.join(OUTPUT_DIR, "top_features_sample.csv"), index=False)
print("Saved top_features_sample.csv")

print("\nEDA complete. Check the folder:", OUTPUT_DIR)

Saved top_features_sample.csv

EDA complete. Check the folder: eda_outputs
